# Laboratorul 6 · Atenția și Transformers

**De la calculul de mână la un model real**  
Lecția 6 · Durată: 90 de minute + partea 4 acasă · Google Colab · CPU

## Obiective
- Implementați atenția în NumPy și verificați-o pe exemple concrete.
- Comparați tokenizarea textului românesc în trei modele.
- Observați probabilitățile următorului token și efectul temperaturii.
- Investigați embeddings contextuale și capete de atenție într-un BERT românesc.

**Cum se folosește:** completați celulele marcate `TODO`, rulați verificările și notați propriile rezultate în tabele și răspunsuri. Codul și valorile obținute nu sunt completate în avans.

| Secțiune | Timp | Statut |
|---|---:|---|
| 1. Atenția în NumPy | 35 min | obligatorie |
| 2. Tokenizarea | 15 min | obligatorie |
| 3. Generarea textului | 30 min | obligatorie |
| 4. Embeddings și vizualizarea atenției | acasă | obligatorie |
| RoGPT2 | după laborator | opțională |

## Înainte de pornire

Folosiți un runtime CPU în Google Colab; nu este necesar GPU. Modelele pot ocupa câteva sute de MB. Rulați celula de descărcare înainte de laborator dacă aveți timp. După închiderea sesiunii Colab, poate fi necesară descărcarea din nou.

Ordinea recomandată: instalați pachetele, importați bibliotecile, apoi rulați celula de descărcare. Secțiunile 1–3 pot fi parcurse în timpul laboratorului; secțiunea 4 se finalizează acasă.

In [1]:
%pip install -q transformers bertviz

In [2]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

np.set_printoptions(precision=3, suppress=True)
print('PyTorch:', torch.__version__)
print('Dispozitiv:', 'GPU' if torch.cuda.is_available() else 'CPU')

PyTorch: 2.11.0+cpu
Dispozitiv: CPU


In [3]:
# Rulați această celulă separat, de preferat înainte de laborator.
MODELE_TOKENIZER = [
    'gpt2',
    'bert-base-multilingual-cased',
    'dumitrescustefan/bert-base-romanian-cased-v1',
]
for nume in MODELE_TOKENIZER:
    AutoTokenizer.from_pretrained(nume)
AutoModelForCausalLM.from_pretrained('gpt2')
AutoModel.from_pretrained('dumitrescustefan/bert-base-romanian-cased-v1')
print('Gata: modelele necesare au fost descărcate.')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (12 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Gata: modelele necesare au fost descărcate.


# 1. Atenția în NumPy

Implementați mecanismul de la zero. În această secțiune, toate proiecțiile și calculele sunt făcute în NumPy.

## 1.1 · Softmax stabil numeric

Completați cele două expresii. Folosiți `axis=-1` și `keepdims=True` pentru a funcționa și pe matrice, rând cu rând.

In [4]:
def softmax(s):
    e = np.exp(s - s.max(axis=-1, keepdims=True))  # TODO: scădeți maximul de pe fiecare rând
    return e / e.sum(axis=-1, keepdims=True)       # TODO: împărțiți la suma de pe fiecare rând

print(softmax(np.array([1.2, 2.8, 0.7])))

[0.152 0.755 0.092]


In [5]:
# Comparați varianta naivă cu funcția voastră pentru scoruri foarte mari.
s = np.array([1000.0, 1001.0])
print('Softmax naiv:', np.exp(s) / np.exp(s).sum())
print('Softmax stabil:', softmax(s))

# Notați observațiile după rulare în răspunsul 1 de la final.

Softmax naiv: [nan nan]
Softmax stabil: [0.269 0.731]


/tmp/ipykernel_1386/2795724686.py:3: RuntimeWarning: overflow encountered in exp
  print('Softmax naiv:', np.exp(s) / np.exp(s).sum())
/tmp/ipykernel_1386/2795724686.py:3: RuntimeWarning: invalid value encountered in divide
  print('Softmax naiv:', np.exp(s) / np.exp(s).sum())


## 1.2 · Atenția pe exemplul din curs

Implementați $QK^T / \sqrt{d_k}$, aplicați masca dacă este transmisă și întoarceți ieșirea împreună cu ponderile.

In [6]:
def attention(Q, K, V, mask=None):
    d_k = K.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)  # TODO: Q @ K.T / sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)
    weights = softmax(scores)
    return weights @ V, weights

In [7]:
# Datele din exemplul rezolvat (Lecția 5, după pasul 7).
q = np.array([[2, 1, 0, 1]])
K = np.array([[0.5, 1.0, 0.3, 0.4],
              [2.0, 1.0, 0.5, 0.6],
              [0.2, 0.6, 0.9, 0.4]])
V = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [0.5, 0.5]])

z, w = attention(q, K, V)
print('Ponderi:', w)
print('z:', z)

Ponderi: [[0.152 0.755 0.092]]
z: [[0.199 0.801]]


**Verificare:** ponderile trebuie să fie aproximativ `[0.152, 0.755, 0.092]`, iar `z` aproximativ `[0.199, 0.801]`. Abaterile mici pot apărea din rotunjirea afișată. Comparați valorile rulate, nu copiați verificarea drept răspuns.

In [8]:
assert np.allclose(w, [[0.152, 0.755, 0.092]], atol=0.002)
assert np.allclose(z, [[0.199, 0.801]], atol=0.002)
print('Verificările pentru exemplul din curs au trecut.')

Verificările pentru exemplul din curs au trecut.


## 1.3 · Toți tokenii deodată, cu mască cauzală

Seed-ul fix face rezultatele comparabile. `True` în mască înseamnă că tokenul din rând poate vedea tokenul din coloană.

In [9]:
rng = np.random.default_rng(42)
X = rng.normal(size=(4, 8))
W_Q, W_K, W_V = (rng.normal(size=(8, 4)) for _ in range(3))

Q, K, V = X @ W_Q, X @ W_K, X @ W_V # TODO: proiectați X cu W_Q, W_K și W_V
mask = np.tril(np.ones((4, 4), dtype=bool))
print(mask)

[[ True False False False]
 [ True  True False False]
 [ True  True  True False]
 [ True  True  True  True]]


In [10]:
Z, W = attention(Q, K, V, mask)
print('Ponderi:\n', W)
print('Suma pe rânduri:', W.sum(axis=1))

assert np.allclose(W.sum(axis=1), 1.0)
assert np.allclose(np.triu(W, k=1), 0.0)
assert np.allclose(W[0], [1.0, 0.0, 0.0, 0.0])
assert np.allclose(W[1], [0.648, 0.352, 0.0, 0.0], atol=0.002)
print('Verificările măștii cauzale au trecut.')

Ponderi:
 [[1.    0.    0.    0.   ]
 [0.648 0.352 0.    0.   ]
 [0.491 0.145 0.363 0.   ]
 [0.285 0.341 0.281 0.093]]
Suma pe rânduri: [1. 1. 1. 1.]
Verificările măștii cauzale au trecut.


Răspundeți la întrebarea 2 de la final: de ce primul token vede doar propria poziție și ce poziții poate vedea tokenul al treilea?

## 1.4 · Multi-head attention cu două capete

Concatenați ieșirile capetelor, apoi aplicați proiecția de ieșire `W_O`.

In [11]:
def multi_head(X, heads, W_O):
    outs = [attention(X @ Wq, X @ Wk, X @ Wv)[0]
            for Wq, Wk, Wv in heads]
    concat = np.concatenate(outs, axis=-1)
    return concat @ W_O  # TODO: proiecția care combină capetele

heads = [tuple(rng.normal(size=(8, 4)) for _ in range(3))
         for _ in range(2)]
W_O = rng.normal(size=(8, 8))
rezultat = multi_head(X, heads, W_O)
print('Dimensiune:', rezultat.shape)
assert rezultat.shape == (4, 8)

Dimensiune: (4, 8)


# 2. Tokenizarea

Comparați GPT-2, BERT multilingv și BERT românesc pe aceleași texte.

In [12]:
nume_modele = {
    'GPT-2': 'gpt2',
    'BERT multilingv': 'bert-base-multilingual-cased',
    'BERT românesc': 'dumitrescustefan/bert-base-romanian-cased-v1',
}
tokenizere = {nume: AutoTokenizer.from_pretrained(model_id)
              for nume, model_id in nume_modele.items()}
fraze = [
    'Banca a aprobat creditul clientului.',
    'Împrumuturile restructurate au dobânzi variabile.',
]
for nume, tok in tokenizere.items():
    for fraza in fraze:
        tokens = tok.tokenize(fraza)
        print(f'{nume:16} {len(tokens):3}  {tokens}')

GPT-2             12  ['B', 'anca', 'Ġa', 'Ġa', 'pro', 'bat', 'Ġcredit', 'ul', 'Ġclient', 'ul', 'ui', '.']
GPT-2             20  ['Ã', 'İ', 'm', 'pr', 'um', 'ut', 'ur', 'ile', 'Ġrestruct', 'urate', 'Ġau', 'Ġdo', 'b', 'Ã¢', 'n', 'zi', 'Ġvari', 'ab', 'ile', '.']
BERT multilingv   10  ['Banca', 'a', 'ap', '##ro', '##bat', 'credit', '##ul', 'client', '##ului', '.']
BERT multilingv   15  ['Î', '##mp', '##rumu', '##turi', '##le', 'rest', '##ru', '##ctura', '##te', 'au', 'dob', '##ân', '##zi', 'variabile', '.']
BERT românesc      6  ['Banca', 'a', 'aprobat', 'creditul', 'clientului', '.']
BERT românesc      8  ['Împrum', '##uturile', 'restruct', '##urate', 'au', 'dobânzi', 'variabile', '.']


### 2.1 · Numărul de tokeni

GPT-2 este tokenizer byte-level; reprezentarea neobișnuită a diacriticelor în lista sa de tokeni este normală. `Ġ` marchează un spațiu înaintea tokenului.

Completați tabelul cu valorile obținute prin rulare.

| Model | Fraza 1 | Fraza 2 |
|---|---:|---:|
| GPT-2 | ['B', 'anca', 'Ġa', 'Ġa', 'pro', 'bat', 'Ġcredit', 'ul', 'Ġclient', 'ul', 'ui', '.'] | ['Ã', 'İ', 'm', 'pr', 'um', 'ut', 'ur', 'ile', 'Ġrestruct', 'urate', 'Ġau', 'Ġdo', 'b', 'Ã¢', 'n', 'zi', 'Ġvari', 'ab', 'ile', '.'] |
| BERT multilingv | ['Banca', 'a', 'ap', '##ro', '##bat', 'credit', '##ul', 'client', '##ului', '.'] |  ['Î', '##mp', '##rumu', '##turi', '##le', 'rest', '##ru', '##ctura', '##te', 'au', 'dob', '##ân', '##zi', 'variabile', '.'] |
| BERT românesc | ['Banca', 'a', 'aprobat', 'creditul', 'clientului', '.'] |  ['Împrum', '##uturile', 'restruct', '##urate', 'au', 'dobânzi', 'variabile', '.'] |

In [14]:
gpt2 = tokenizere['GPT-2']
n_ro = len(gpt2.tokenize('Banca a aprobat creditul clientului.'))
n_en = len(gpt2.tokenize("The bank approved the client's loan."))
print('Tokeni română, engleză, raport:', n_ro, n_en, round(n_ro / n_en, 2))

# Pentru costul unui milion de fraze, folosiți prețul de 1 USD / milion de tokeni.

Tokeni română, engleză, raport: 12 8 1.5


### 2.3 · Diacritice cu virgulă și cu sedilă

`ș, ț` cu virgulă sunt caractere diferite de `ş, ţ` cu sedilă. Comparați segmentarea și notați ce normalizare ați aplica înainte de a trimite textele unui model.

In [15]:
ro = tokenizere['BERT românesc']
print(ro.tokenize('știință și țară'))  # virgulă
print(ro.tokenize('ştiinţă şi ţară'))  # sedilă

['știință', 'și', 'țară']
['[UNK]', '[UNK]', '[UNK]']


# 3. Generarea textului

GPT-2 este un model decoder-only antrenat în principal pe texte în engleză. În această parte, urmărim distribuția următorului token și generăm token cu token.

In [16]:
tok = tokenizere['GPT-2']
model = AutoModelForCausalLM.from_pretrained('gpt2')
model.eval()

def top10(model, tok, text):
    ids = tok(text, return_tensors='pt').input_ids
    with torch.no_grad():
        logits = model(ids).logits[0, -1]
    print('Dimensiunea vectorului logits:', logits.shape)
    probs = torch.softmax(logits, dim=-1)
    print('Suma probabilităților:', probs.sum().item())
    top = torch.topk(probs, 10)
    for p, i in zip(top.values, top.indices):
        print(f'{tok.decode(int(i))!r:16} {p.item():.3f}')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [17]:
top10(model, tok, 'The bank approved the')
# Notați dimensiunea logits și primii trei tokeni cu probabilitățile lor în răspunsul 5.

Dimensiunea vectorului logits: torch.Size([50257])
Suma probabilităților: 1.000011920928955
' loan'          0.066
' deal'          0.066
' sale'          0.037
' transaction'   0.033
' plan'          0.025
' purchase'      0.024
' new'           0.019
' $'             0.019
' move'          0.016
' settlement'    0.012


## 3.2 · Generare manuală

Completați alegerea greedy și împărțirea logits la temperatură. Temperatura se aplică înainte de Softmax.

In [ ]:
def genereaza(model, tok, text, n_tokeni=25, temperatura=1.0, greedy=False):
    ids = tok(text, return_tensors='pt').input_ids
    for _ in range(n_tokeni):
        with torch.no_grad():
            logits = model(ids).logits[0, -1]
        if greedy:
            urm = torch.argmax(logits)  # TODO: tokenul cu scor maxim
        else:
            probs = torch.softmax(logits / temperatura, dim=-1)  # TODO: temperatura
            urm = torch.multinomial(probs, 1)[0]
        ids = torch.cat([ids, urm.view(1, 1)], dim=1)
    return tok.decode(ids[0])

In [21]:
text = 'The bank approved the'
print('GREEDY:', genereaza(model, tok, text, greedy=True))
for T in [0.2, 0.7, 1.5]:
    for incercare in range(2):
        print(f'T = {T}:', genereaza(model, tok, text, temperatura=T))

GREEDY: The bank approved the loan in May, but the bank has not yet approved the loan.

The bank said it will not be able to
T = 0.2: The bank approved the loan in May, but the bank's chief executive, Mark Carney, said the bank was not aware of the loan's terms
T = 0.2: The bank approved the deal, which will allow the bank to continue to operate in the country.

The bank said it would not be able
T = 0.7: The bank approved the deal at a meeting on Sunday morning, the official said.

About 50 traders and lots of speculators have signed up
T = 0.7: The bank approved the sale to an unnamed buyer.

The problem is, it's not clear whether the bank's current owners ever sold out
T = 1.5: The bank approved the sober classified element, revealing most Bank officials using revelation offered schools dependent on bail outs stated offical tales pharma breaksussie firm
T = 1.5: The bank approved the Sharon blessing costing approx $15'); who can read but 249 agony prods right next without jiu boxing

Completați tabelul pe baza textelor generate. Comparați varietatea, coerența și repetițiile.

| Setare | Observațiile mele |
|---|---|
| Greedy | Cel mai bine seamănă cu prima variantă a T=0.2 |
| T = 0.2 | Lungime medie a textului generat |
| T = 0.7 | Cea mai mică lungime medie a textului generat |
| T = 1.5 | Cea mai mică mare medie a textului generat |

Folosiți aceste observații în răspunsul 6.

### Experiment opțional · RoGPT2

Modelul este mai vechi și poate genera română imperfectă; aici contează mecanismul, nu calitatea textului. Descărcarea modelului este separată de pregătirea obligatorie.

In [22]:
tok_ro_gpt = AutoTokenizer.from_pretrained('readerbench/RoGPT2-base')
model_ro_gpt = AutoModelForCausalLM.from_pretrained('readerbench/RoGPT2-base')
model_ro_gpt.eval()
top10(model_ro_gpt, tok_ro_gpt, 'Banca a aprobat')
print(genereaza(model_ro_gpt, tok_ro_gpt, 'Banca a aprobat', temperatura=0.7))

config.json:   0%|          | 0.00/814 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/985k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/542k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  510MB            

... (3 outputs left out)

Dimensiunea vectorului logits: torch.Size([50257])
Suma probabilităților: 1.0000123977661133
' în'            0.108
' un'            0.075
','              0.060
' o'             0.050
'-'              0.049
' finanțarea'    0.013
' planul'        0.013
' pentru'        0.013
' la'            0.012
' și'            0.010


model.safetensors: reconstructing file:   0%|          |  0.00B /  510MB            

model.safetensors: downloading bytes:           |  0.00B            

Banca a aprobat o lege prin care „poate fi folosit un risc mai mare”, deoarece un risc de credit este mai mare atunci când ratele dobânzilor


# 4. Embeddings contextuale și vizualizarea atenției

Această secțiune este obligatorie și se poate finaliza acasă. În BERT-ul românesc, stratul 0 este embedding-ul inițial cu informație pozițională, iar stratul 12 este ieșirea ultimului bloc.

In [23]:
import torch.nn.functional as Fn

nume_ro = 'dumitrescustefan/bert-base-romanian-cased-v1'
tok_bert = AutoTokenizer.from_pretrained(nume_ro)
bert = AutoModel.from_pretrained(
    nume_ro, output_hidden_states=True, attn_implementation='eager'
)
bert.eval()

def vector_cuvant(fraza, cuvant, strat):
    enc = tok_bert(fraza, return_tensors='pt')
    with torch.no_grad():
        out = bert(**enc)
    tokens = tok_bert.convert_ids_to_tokens(enc.input_ids[0])
    parti = tok_bert.tokenize(cuvant)
    for i in range(len(tokens) - len(parti) + 1):
        if tokens[i:i + len(parti)] == parti:
            return out.hidden_states[strat][0, i:i + len(parti)].mean(dim=0)
    raise ValueError(f'Nu am găsit {cuvant!r} în: {fraza}')

fraze_banca = [
    'Am deschis un cont la bancă.',
    'Am luat un credit de la bancă.',
    'M-am așezat pe o bancă în parc.',
    'Bătrânul stă pe bancă lângă fântână.',
]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [24]:
for strat in [0, 12]:
    vecs = torch.stack([vector_cuvant(f, 'bancă', strat) for f in fraze_banca])
    sim = Fn.cosine_similarity(vecs.unsqueeze(1), vecs.unsqueeze(0), dim=-1)
    print(f'Stratul {strat}:')
    print(sim.numpy().round(3))

Stratul 0:
[[1.    0.971 0.971 0.961]
 [0.971 1.    1.    0.954]
 [0.971 1.    1.    0.954]
 [0.961 0.954 0.954 1.   ]]
Stratul 12:
[[1.    0.953 0.659 0.594]
 [0.953 1.    0.625 0.56 ]
 [0.659 0.625 1.    0.832]
 [0.594 0.56  0.832 1.   ]]


Urmăriți dacă la stratul 12 propozițiile 1 și 2 (sens financiar) sunt mai apropiate între ele decât de propozițiile 3 și 4 (loc de șezut). Nu vă așteptați la o separare perfectă. Notați valorile pentru perechile 1–2 și 1–3 la straturile 0 și 12.

| Strat | Similaritate 1–2 | Similaritate 1–3 | Ce observ? |
|---:|---:|---:|---|
| 0 | 0.971 | 0.971 | Similaritățile sunt egale; la stratul inițial, contextul nu diferențiază încă sensurile. |
| 12 | 0.953 | 0.659 | Perechile cu sens financiar sunt mai apropiate, dar separarea nu este perfectă. |

## 4.2 · Vizualizarea capetelor de atenție

În vizualizare, alegeți stratul și tokenul. Căutați un cap unde `aprobat` urmărește `creditul`, apoi un cap al cărui comportament nu îl puteți interpreta. Înregistrați stratul și numărul capului.

In [25]:
from bertviz import head_view

bert_att = AutoModel.from_pretrained(
    nume_ro, output_attentions=True, attn_implementation='eager'
)
enc = tok_bert('Banca a aprobat creditul clientului.', return_tensors='pt')
with torch.no_grad():
    att = bert_att(**enc).attentions
head_view(att, tok_bert.convert_ids_to_tokens(enc.input_ids[0]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: dumitrescustefan/bert-base-romanian-cased-v1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


<IPython.core.display.Javascript object>

**Notează aici**

- Capul în care `aprobat` se uită mult la `creditul`: stratul `1` cu capul `roșu` sau stratul `2` cu capul `albastru`.
- Observația mea: Începând cu stratul 4 toate capetele acordă mare atenție pentru `[SEP]`, iar la statul `0` nu există legături strânse pentru capete (ce este de aștepatt pentru primul nivel de inițiere)

# 5. Întrebări de sinteză

Răspundeți cu valorile obținute în notebook și cu formulări proprii. Lăsați răspunsurile aici sau adăugați celule Markdown sub fiecare întrebare.

1. **Softmax (1.1):** Ce a produs varianta naivă pentru `[1000, 1001]` și ce a produs varianta stabilă? De ce apare diferența și de ce scăderea maximului nu schimbă rezultatul corect?

   Răspuns: Varianta naivă a produs `[nan, nan]`, deoarece `exp(1000)` și `exp(1001)` depășesc limita numerică. Varianta stabilă a produs `[0.269, 0.731]`. Scăderea maximului nu schimbă rezultatul: introduce același factor de multiplicare în numărător și numitor, iar acesta se reduce.

2. **Masca (1.3):** De ce primul rând al ponderilor mascate este `[1, 0, 0, 0]`? Ce tokeni vede al treilea token?

   Răspuns: Primul rând permite atenția numai către primul token; softmax-ul unei singure valori permise este `1`, deci rândul devine `[1, 0, 0, 0]`. Al treilea token vede primele trei poziții: pe sine și pe cei doi tokeni anteriori. În execuția mea, ponderile au fost `[0.491, 0.145, 0.363, 0]`.

3. **Tokenizarea (2.2):** De câte ori mai mulți tokeni are fraza românească decât cea engleză în GPT-2? La 1 USD / milion de tokeni, cât costă un milion de fraze în fiecare limbă?

   Răspuns: GPT-2 a împărțit fraza românească în `12` tokeni și pe cea engleză în `8`, deci româna a folosit de `1,5 `ori mai mulți tokeni. La `1 USD` pentru un milion de tokeni, un milion de fraze ar costa aproximativ `12 USD` în română și `8 USD` în engleză.

4. **Diacritice (2.3):** Ce diferență ați observat între `știință` și `ştiinţă`? Cum ar trebui pregătit un set de texte românești înainte de utilizare?

   Răspuns: `știință` și `țară`, cu virgulă, a fost tokenizat în `['știință', 'și', 'țară']`. Varianta cu sedilă a devenit `['[UNK]', '[UNK]', '[UNK]']`. Înainte de utilizare, textele trebuie normalizate consecvent la diacriticele românești corecte, cu virgulă.

5. **Logits (3.1):** Ce dimensiune are vectorul logits la GPT-2, ce reprezintă și care au fost primii trei tokeni propuși cu probabilitățile lor?

   Răspuns: Vectorul are dimensiunea `50 257`, adică modelul are câte un scor pentru fiecare token din vocabularul GPT-2. Primii trei tokeni afișați au fost `' loan'` – `0,066`, `' deal'` – `0,066` și `' sale'` – `0,037`.

6. **Temperatură (3.2):** De ce greedy produce același text? Ce se schimbă când temperatura crește și de ce, conform formulei Softmax?

   Răspuns: Greedy alege de fiecare dată tokenul cu scorul cel mai mare; pentru același prompt, selecția este deterministă. La `T = 0,2`, exemplele au fost coerente și asemănătoare; la `T = 0,7`, au fost coerente, dar mai variate; la `T = 1,5`, au devenit mult mai aleatorii și incoerente. O temperatură mai mare aplatizează distribuția Softmax și face tokenii cu probabilitate mai mică mai ușor de ales.

7. **Embeddings (4):** Ce similarități ați obținut între propozițiile 1–2 și 1–3 la straturile 0 și 12? Ce indică numerele?

   Răspuns: La stratul `0`, similaritățile pentru perechile `1–2` și `1–3` au fost ambele `0,971`. La stratul `12` au fost `0,953`, respectiv `0,659`. Reprezentările de la stratul final reflectă mai bine contextul financiar față de sensul de `„bancă”` pentru șezut, dar nu separă complet sensurile. Similaritatea dintre propozițiile `3` și `4` este `0,832`, ceea ce se potrivește cu sensul comun de loc pentru șezut.

# 6. Evaluare și predare

Ponderile de mai jos păstrează proporțional contribuțiile din documentul inițial și sunt normalizate la 100%. Ultima valoare este rotunjită astfel încât totalul afișat să fie exact 100%.

| Criteriu | Pondere normalizată |
|---|---:|
| Partea 1: cod funcțional și verificările trecute                              | 40%                    |
| Partea 2: tabelul completat și răspunsurile 3–4                               | 20%                    |
| Partea 3: cod funcțional, tabelul de observații și răspunsurile 5–6           | 30%                    |
| Claritatea răspunsurilor (cu numere concrete, formulate cu propriile cuvinte) | 10%                    |
| Partea 4  (la dorință)                                                        | 10% (bonus)            |

**De predat:** notebook-ul rulat (`File → Download → Download .ipynb`) și răspunsurile cu rezultate proprii. Verificați înainte de predare că toate celulele au rulat și că tabelele și întrebările sunt completate.